# Parallel Corpora Creation

This notebook creates parallel corpora for specific language pairs needed for similarity transfer experiments.

## Language Pairs

| Pair | Role | Languages |
|------|------|-----------|
| en–tl | baseline + distant | English, Tagalog |
| war–tl | similar donor | Waray, Tagalog |
| en–war | baseline + distant | English, Waray |
| ceb–war | similar donor | Cebuano, Waray |

## Imports

In [12]:
import json
from pathlib import Path
import pandas as pd

## Configuration

Define language pairs and mappings.

In [13]:
# Language code mappings
lang_code_to_name = {
    "en": "english",
    "tl": "tagalog",
    "war": "waray",
    "ceb": "cebuano"
}

# Define language pairs with their roles
language_pairs = [
    {
        "pair": "en-tl",
        "role": "baseline + distant",
        "src_code": "en",
        "tgt_code": "tl",
        "src_lang": "english",
        "tgt_lang": "tagalog"
    },
    {
        "pair": "war-tl",
        "role": "similar donor",
        "src_code": "war",
        "tgt_code": "tl",
        "src_lang": "waray",
        "tgt_lang": "tagalog"
    },
    {
        "pair": "en-war",
        "role": "baseline + distant",
        "src_code": "en",
        "tgt_code": "war",
        "src_lang": "english",
        "tgt_lang": "waray"
    },
    {
        "pair": "ceb-war",
        "role": "similar donor",
        "src_code": "ceb",
        "tgt_code": "war",
        "src_lang": "cebuano",
        "tgt_lang": "waray"
    }
]

print(f"Total language pairs: {len(language_pairs)}")
for pair_info in language_pairs:
    print(f"  {pair_info['pair']:10} | {pair_info['role']:20} | {pair_info['src_lang']:12} → {pair_info['tgt_lang']}")

Total language pairs: 4
  en-tl      | baseline + distant   | english      → tagalog
  war-tl     | similar donor        | waray        → tagalog
  en-war     | baseline + distant   | english      → waray
  ceb-war    | similar donor        | cebuano      → waray


## Load Processed Data

Load the cleaned data from the preprocessing step.

In [ ]:
processed_dir = Path("../data/processed")

# Load all language data
language_data = {}
for lang_name in ["english", "tagalog", "waray", "cebuano"]:
    json_file = processed_dir / f"{lang_name}_clean.json"
    with open(json_file, "r", encoding="utf-8") as f:
        language_data[lang_name] = json.load(f)
    print(f"Loaded {lang_name}: {len(language_data[lang_name])} verses")

print(f"\nTotal languages loaded: {len(language_data)}")

Loaded english: 7810 verses
Loaded tagalog: 7810 verses


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\processed\\bikolano_clean.json'

## Create Parallel Corpora

Generate parallel text files for each language pair.

In [ ]:
def create_parallel_corpus(src_data, tgt_data, pair_name, output_dir):
    """
    Create parallel corpus files for a language pair.
    
    Args:
        src_data: List of verse dictionaries for source language
        tgt_data: List of verse dictionaries for target language
        pair_name: Name of the language pair (e.g., 'en-tl')
        output_dir: Directory to save the parallel corpus files
    
    Returns:
        Dictionary with statistics about the created corpus
    """
    # Verify that both datasets have the same length and alignment
    if len(src_data) != len(tgt_data):
        raise ValueError(f"Data length mismatch: {len(src_data)} vs {len(tgt_data)}")
    
    # Create output directory
    pair_dir = output_dir / pair_name
    pair_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract language codes from pair name
    src_code, tgt_code = pair_name.split("-")
    
    # Write source file
    src_file = pair_dir / f"{pair_name}.{src_code}"
    with open(src_file, "w", encoding="utf-8") as f:
        for verse in src_data:
            f.write(f"{verse['text']}\n")
    
    # Write target file
    tgt_file = pair_dir / f"{pair_name}.{tgt_code}"
    with open(tgt_file, "w", encoding="utf-8") as f:
        for verse in tgt_data:
            f.write(f"{verse['text']}\n")
    
    # Write reference file (for traceability)
    ref_file = pair_dir / f"{pair_name}.ref"
    with open(ref_file, "w", encoding="utf-8") as f:
        for verse in src_data:
            f.write(f"{verse['reference']}\n")
    
    # Calculate statistics
    stats = {
        "pair": pair_name,
        "num_lines": len(src_data),
        "src_words": sum(len(v['text'].split()) for v in src_data),
        "tgt_words": sum(len(v['text'].split()) for v in tgt_data),
        "src_file": str(src_file),
        "tgt_file": str(tgt_file),
        "ref_file": str(ref_file)
    }
    
    return stats

# Create output directory
parallel_dir = Path("../data/parallel")
parallel_dir.mkdir(parents=True, exist_ok=True)

# Create parallel corpora for all language pairs
corpus_stats = []

print("Creating parallel corpora...")
print("=" * 80)

for pair_info in language_pairs:
    pair_name = pair_info["pair"]
    src_lang = pair_info["src_lang"]
    tgt_lang = pair_info["tgt_lang"]
    
    print(f"\nCreating {pair_name} ({src_lang} → {tgt_lang})...")
    
    src_data = language_data[src_lang]
    tgt_data = language_data[tgt_lang]
    
    stats = create_parallel_corpus(src_data, tgt_data, pair_name, parallel_dir)
    corpus_stats.append({**pair_info, **stats})
    
    print(f"  ✓ Created {stats['num_lines']} parallel sentences")
    print(f"  ✓ Source words: {stats['src_words']:,}")
    print(f"  ✓ Target words: {stats['tgt_words']:,}")

print("\n" + "=" * 80)
print("✓ All parallel corpora created successfully!")
print("=" * 80)

## Summary Statistics

Display comprehensive statistics about all created parallel corpora.

In [ ]:
# Create summary DataFrame
df_stats = pd.DataFrame(corpus_stats)

print("\n" + "=" * 100)
print("PARALLEL CORPORA SUMMARY")
print("=" * 100)

print("\nBy Language Pair:")
print("-" * 100)
for _, row in df_stats.iterrows():
    print(f"\n{row['pair'].upper()} ({row['role']})")
    print(f"  Languages: {row['src_lang'].capitalize()} → {row['tgt_lang'].capitalize()}")
    print(f"  Sentences: {row['num_lines']:,}")
    print(f"  Source words: {row['src_words']:,} | Avg per sentence: {row['src_words']/row['num_lines']:.1f}")
    print(f"  Target words: {row['tgt_words']:,} | Avg per sentence: {row['tgt_words']/row['num_lines']:.1f}")

print("\n" + "-" * 100)
print("\nBy Role:")
print("-" * 100)

baseline_pairs = df_stats[df_stats['role'] == 'baseline + distant']
donor_pairs = df_stats[df_stats['role'] == 'similar donor']

print(f"\nBaseline + Distant pairs ({len(baseline_pairs)}):")
for _, row in baseline_pairs.iterrows():
    print(f"  • {row['pair']:10} | {row['src_lang']:12} → {row['tgt_lang']:12} | {row['num_lines']:,} sentences")

print(f"\nSimilar Donor pairs ({len(donor_pairs)}):")
for _, row in donor_pairs.iterrows():
    print(f"  • {row['pair']:10} | {row['src_lang']:12} → {row['tgt_lang']:12} | {row['num_lines']:,} sentences")

print("\n" + "=" * 100)
print(f"Total parallel corpora: {len(corpus_stats)}")
print(f"Total sentences per corpus: {df_stats['num_lines'].iloc[0]:,}")
print(f"Output directory: {parallel_dir.absolute()}")
print("=" * 100)

## Save Metadata

Save corpus metadata for future reference.

In [ ]:
# Save metadata as JSON
# Convert numpy types to Python types for JSON serialization
corpus_stats_json = []
for stat in corpus_stats:
    stat_copy = stat.copy()
    stat_copy['num_lines'] = int(stat_copy['num_lines'])
    stat_copy['src_words'] = int(stat_copy['src_words'])
    stat_copy['tgt_words'] = int(stat_copy['tgt_words'])
    corpus_stats_json.append(stat_copy)

metadata = {
    "description": "Parallel corpora for Philippine low-resource language MT with similarity transfer",
    "creation_date": "2025-11-08",
    "source": "Bible (Matthew, Mark, Luke)",
    "total_pairs": len(corpus_stats_json),
    "sentences_per_corpus": int(df_stats['num_lines'].iloc[0]),
    "language_pairs": corpus_stats_json
}

metadata_file = parallel_dir / "corpus_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"✓ Metadata saved to {metadata_file}")

# Save statistics as CSV
csv_file = parallel_dir / "corpus_statistics.csv"
df_stats[['pair', 'role', 'src_lang', 'tgt_lang', 'num_lines', 'src_words', 'tgt_words']].to_csv(
    csv_file, index=False
)

print(f"✓ Statistics saved to {csv_file}")

## Create TSV/CSV Versions (Alternative Format)

Create TSV versions of the parallel corpora for frameworks that prefer this format.

In [ ]:
# Create TSV and CSV versions for each language pair
print("Creating TSV/CSV versions...")
print("=" * 80)

for pair_info in language_pairs:
    pair_name = pair_info["pair"]
    src_lang = pair_info["src_lang"]
    tgt_lang = pair_info["tgt_lang"]
    src_code = pair_info["src_code"]
    tgt_code = pair_info["tgt_code"]
    
    pair_dir = parallel_dir / pair_name
    
    src_data = language_data[src_lang]
    tgt_data = language_data[tgt_lang]
    
    # Create TSV file (tab-separated)
    tsv_file = pair_dir / f"{pair_name}.tsv"
    with open(tsv_file, "w", encoding="utf-8") as f:
        # Write header
        f.write(f"{src_code}\t{tgt_code}\n")
        # Write data
        for src_verse, tgt_verse in zip(src_data, tgt_data):
            f.write(f"{src_verse['text']}\t{tgt_verse['text']}\n")
    
    # Create CSV file (comma-separated, with proper escaping)
    csv_file = pair_dir / f"{pair_name}.csv"
    df_pair = pd.DataFrame({
        src_code: [v['text'] for v in src_data],
        tgt_code: [v['text'] for v in tgt_data]
    })
    df_pair.to_csv(csv_file, index=False, encoding='utf-8')
    
    print(f"✓ Created {pair_name}.tsv and {pair_name}.csv")

print("\n" + "=" * 80)
print("✓ All TSV/CSV files created successfully!")
print("\n📁 Each language pair now has:")
print("  • Separate text files (.src, .tgt) - for fairseq, OpenNMT, Transformers")
print("  • TSV file (.tsv) - tab-separated with header")
print("  • CSV file (.csv) - comma-separated with proper escaping")
print("  • Reference file (.ref) - verse references for traceability")
print("=" * 80)

## Verify Alignment

Verify that parallel files are properly aligned by checking a sample.

In [ ]:
# Verify alignment for a sample pair
sample_pair = language_pairs[0]  # en-tl
pair_name = sample_pair['pair']
src_code = sample_pair['src_code']
tgt_code = sample_pair['tgt_code']

pair_dir = parallel_dir / pair_name

# Read first 5 lines from each file
src_file = pair_dir / f"{pair_name}.{src_code}"
tgt_file = pair_dir / f"{pair_name}.{tgt_code}"
ref_file = pair_dir / f"{pair_name}.ref"

with open(src_file, "r", encoding="utf-8") as f:
    src_lines = [line.strip() for line in f.readlines()[:5]]

with open(tgt_file, "r", encoding="utf-8") as f:
    tgt_lines = [line.strip() for line in f.readlines()[:5]]

with open(ref_file, "r", encoding="utf-8") as f:
    ref_lines = [line.strip() for line in f.readlines()[:5]]

print(f"Alignment Verification for {pair_name.upper()}")
print("=" * 100)

for i, (ref, src, tgt) in enumerate(zip(ref_lines, src_lines, tgt_lines), 1):
    print(f"\nSample {i}: {ref}")
    print(f"  {src_code.upper()}: {src}")
    print(f"  {tgt_code.upper()}: {tgt}")

print("\n" + "=" * 100)
print("✓ Alignment verified successfully!")